#### Model train and evaluation

In [1]:
# requirements
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

##### load and prepare data

In [2]:
# indexing
SOS_token = 0
EOS_token = 1

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [3]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [4]:
# read data
def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")

    # Read the file and split into lines
    lines = open('data 2/%s-%s.txt' % (lang1, lang2), encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs, make Lang instances
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

In [5]:
# filter data by length and content
MAX_LENGTH = 10

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [6]:
def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

input_lang, output_lang, pairs = prepareData('eng', 'fra', True)
print(random.choice(pairs))

Reading lines...
Read 135842 sentence pairs
Trimmed to 11445 sentence pairs
Counting words...
Counted words:
fra 4601
eng 2991
['vous etes tous des laches', 'you re all cowards']


In [7]:
# load and define pre-trained word2vec model

import gensim.downloader as api

# load word2vec model
pretrained_word2vec = api.load('word2vec-google-news-300')
embed_dim = 300
VOCAB_SIZE=11445

def build_embed_matrix(lang, word2vec, embed_dim=300):
    std = float(word2vec.vectors.std())
    embed_matrix = np.zeros((VOCAB_SIZE, embed_dim))
    for word, id in lang.word2index.items():
        if word in word2vec:
            embed_matrix[id] = word2vec[word]
        else:
            embed_matrix[id] = np.random.normal(scale=std, size=embed_dim)

    for spec_id in (SOS_token, EOS_token):
        embed_matrix[spec_id] = np.random.normal(scale=std, size=embed_dim)

    return torch.FloatTensor(embed_matrix)

out_embed_wts = build_embed_matrix(output_lang, pretrained_word2vec, embed_dim)

[==================================================] 100.0% 1662.8/1662.8MB downloaded


##### encoder

In [46]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.gru(embedded)
        return output, hidden

##### decoder

In [47]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, pretrained_embed):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding.from_pretrained(pretrained_embed, hidden_size)
        self.gru = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.out(output)
        return output, hidden

##### attention decoder

In [53]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights

class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, pretrained_embed, num_layers=1, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        embed_size = pretrained_embed.size(1)
        self.embedding = nn.Embedding.from_pretrained(pretrained_embed.clone(), freeze=False)
        self.attention = BahdanauAttention(hidden_size)
        self.gru = nn.GRU(input_size=embed_size+hidden_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions


    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        query = hidden[-1].unsqueeze(1)
        context, attn_weights = self.attention(query, encoder_outputs)
        input_gru = torch.cat((embedded, context), dim=2)

        output, hidden = self.gru(input_gru, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

##### train

In [11]:
# prepare training data
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData('eng', 'fra', True)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

In [12]:
# train the model
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [13]:
# print time elapsed and estimated time remaining
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [14]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [15]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

##### evaluate

In [16]:

def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [17]:
# evaluate random sentences
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

##### network initialization (baseline, w/ attention, and w/ pretrained embeddings)

In [ ]:
# 1. initalize baseline network and begin training

hidden_size = 128
batch_size = 32

# 1. baseline 

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, 80, print_every=5, plot_every=5)

encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

In [ ]:
# 1b. baseline w/ attention

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, 80, print_every=5, plot_every=5)

encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

In [ ]:
# 2. w/ decoder pretrained embeddings
input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words, out_embed_wts).to(device)

train(train_dataloader, encoder, decoder, 80, print_every=5, plot_every=5)

encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

#### 2.

In [49]:
# define the baseline and experiment configurations
experiments = [('baseline (128, 1)', 128, 1),
               ('more units (256, 1)', 256, 1),
               ('more layers (128, 2)', 128, 2)]

In [54]:
# define experiment training
def train_experiment(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100):
    start = time.time()
    epoch_losses = []
    print_loss_total = 0  # Reset every print_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)

        epoch_losses.append(loss)
        print_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

    return epoch_losses, time.time() - start


In [56]:
# generate experiment results
results = {}
for experiment, hidden_size, num_layers in experiments:
    print(f'\n---- {experiment} ----')
    torch.manual_seed(42)

    encoder = EncoderRNN(input_lang.n_words, hidden_size, num_layers).to(device)
    decoder = AttnDecoderRNN(hidden_size, output_lang.n_words, out_embed_wts, num_layers).to(device)

    losses, time_elapsed = train_experiment(train_dataloader, encoder, decoder, n_epochs=20)

    encoder.eval()
    decoder.eval()
    evaluateRandomly(encoder, decoder)

    results[experiment] = {"losses": losses, "time": time_elapsed,
                           "hidden size": hidden_size, "layers": num_layers}


---- baseline (128, 1) ----
> nous n allons nulle part
= we re not going anywhere
< we re not going anywhere <EOS>

> je suis le premier musicien de la famille
= i am the first musician in my family
< i am the first musician in my family <EOS>

> je suis pret a jeter l eponge
= i m ready to throw in the towel
< i am ready to throw in the towel <EOS>

> il a peur de faire des erreurs
= he is afraid of making mistakes
< he is afraid of making mistakes <EOS>

> je suis plus astucieuse que vous
= i m smarter than you
< i m smarter than you <EOS>

> je suis impulsive
= i m impulsive
< i m impulsive <EOS>

> je me rends en ville
= i m going downtown
< i m going downtown <EOS>

> je suis dans la maison
= i m in the house
< i m in the house <EOS>

> je ne vais pas t epouser
= i m not marrying you
< i m not marrying you <EOS>

> ils sont en salle des professeurs
= they are in the teachers room
< they are in the teachers room <EOS>


---- more units (256, 1) ----
> je suis forte
= i m strong
< 

In [60]:
# compare experiment results in a table
base_experiment = experiments[0][0]
base = results[experiment]
base_final = base['losses'][-1]

rows = []
for experiment_name, result in results.items():
    rows.append({"experiment": experiment_name,
                 "hidden size": result['hidden size'],
                 "layers": result['layers'],
                 "time (s)": result['time'],
                 'time / baseline': result['time'] / base['time'],
                 'final loss': result['losses'][-1],
                 'best loss': min(result['losses']),
                 })

comparison_df = pd.DataFrame(rows).set_index('experiment')
pd.options.display.float_format = '{:.2f}'.format
display(comparison_df)

,hidden size,layers,time (s),time / baseline,final loss,best loss
experiment,,,,,,
"baseline (128, 1)",128,1,242.22,0.85,0.10,0.10
"more units (256, 1)",256,1,369.76,1.30,0.04,0.04
"more layers (128, 2)",128,2,284.34,1.00,0.17,0.17


#### Discussion of 3 models' comparison results
The model with double the baseline's number of hidden units (256 versus 128) returned the lowest and best training loss among the three model configurations; its english translation predictions were closest to the target, or actual, translations. Its higher number of hidden units likely enabled the model to capture more complex information and patterns in the text data. However, its hidden units also required a higher number of parameters to train. The training time under the `time (s)` column of the table reflects this: this model took the longest time to train among the three models. </br></br>
Interestingly, despite the two models sharing the same number of hidden units, the model with two layers returned a worst best training loss compared to the baseline model's best training loss. A potential explanation for this is that the 2-layer model suffered from the vanishing gradient problem during backpropagation due to its added layer. Another cause for its worse training loss may be that the optimizer's learning rate was unadjusted (i.e., it was too high or too low), and so the optimizer struggled to converge on a minimimum. </br></br>
Looking at the three models' performance on random sentences each model did a decent job of correctly predicting the english translation of a given french sentence. Though several inaccurate predictions are seen under the model with 128 hidden units and 2 layers. Important to note is that this observation reflects only the one set of randomly selected translations (see `outputs/3_experiments_output.md`).